In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


aqicat = pd.read_csv("./AQI_Category_Historical.csv")
pollutants = pd.read_csv("./Pollutant_Historical.csv")
stations = pd.read_csv("./Station_Details_Historical.csv")
climate = pd.read_csv("Climate_Conditions_Historical.csv")
climate


,Station_id,timestamp,wind_speed_and_direction,temperature,precipitation,relative_humidity
0,ST-AR-001,2026-02-01 00:00:00,"3.2 km/h, 317 deg",-11.1,0.0,95
1,ST-AR-001,2026-02-01 01:00:00,"3.4 km/h, 317 deg",-11.4,0.0,95
2,ST-AR-001,2026-02-01 02:00:00,"3.3 km/h, 319 deg",-11.2,0.0,94
3,ST-AR-001,2026-02-01 03:00:00,"3.2 km/h, 322 deg",-10.8,0.0,94
4,ST-AR-001,2026-02-01 04:00:00,"2.8 km/h, 325 deg",-12.2,0.0,93
...,...,...,...,...,...,...
88027,ST-TR-131,2026-02-28 19:00:00,"8.2 km/h, 188 deg",22.1,0.0,47
88028,ST-TR-131,2026-02-28 20:00:00,"7.3 km/h, 210 deg",21.0,0.0,53
88029,ST-TR-131,2026-02-28 21:00:00,"5.2 km/h, 182 deg",20.1,0.0,56
88030,ST-TR-131,2026-02-28 22:00:00,"7.6 km/h, 149 deg",18.8,0.0,62


In [4]:
df_master = aqicat
df_master=df_master.merge(pollutants, on=['Station_id', 'timestamp'], how='inner')
df_master.drop(columns=['category_name','Health_advisory',],inplace=True)
df_master=df_master.merge(climate, on=['Station_id', 'timestamp'], how='inner')
df_master
df_master['timestamp'] = pd.to_datetime(df_master['timestamp'])

df_master = df_master.sort_values(by = 'timestamp')
df_master['wind_speed'] = df_master['wind_speed_and_direction'].str.split(',',expand=True)[0]
df_master['wind_speed_num'] = df_master['wind_speed'].str.extract(r'(\d+\.?\d*)').astype(float)
df_master['hours'] = df_master['timestamp'].dt.hour
df_master['month'] = df_master['timestamp'].dt.month
df_master['day_of_week'] = df_master['timestamp'].dt.dayofweek
df_master = df_master.drop(columns=['wind_speed_and_direction','wind_speed','Range','Station_id','timestamp'])
df_master

,Exact_AQI,PM2_5,PM10,CO,NO2,temperature,precipitation,relative_humidity,wind_speed_num,hours,month,day_of_week
0,59,15.8,17.6,203.0,11.9,-11.1,0.0,95,3.2,0,2,6
6048,52,14.3,17.5,223.0,1.2,7.1,0.1,87,4.2,0,2,6
53760,135,51.1,52.6,409.0,17.2,6.3,0.0,100,0.8,0,2,6
77280,156,73.1,77.1,798.0,18.4,11.9,0.0,98,4.4,0,2,6
34272,155,82.2,87.3,753.0,16.7,15.2,0.0,87,3.0,0,2,6
...,...,...,...,...,...,...,...,...,...,...,...,...
20831,163,92.0,99.4,555.0,3.7,17.1,2.1,97,6.7,23,2,5
66527,111,54.7,68.0,523.0,10.5,10.1,0.0,74,2.5,23,2,5
21503,164,106.9,127.6,468.0,12.3,18.5,0.0,79,10.4,23,2,5
58463,152,54.8,74.7,289.0,9.0,13.1,0.0,68,1.5,23,2,5


In [5]:
x_train,x_test,y_train,y_test = train_test_split(df_master.drop(columns=['Exact_AQI']),df_master['Exact_AQI'],test_size=0.2,shuffle=True)



In [7]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,        
    learning_rate=0.05,     
    max_depth=6,             
    random_state=42,
    early_stopping_rounds=50
)

xgb_model.fit(x_train,y_train,eval_set=[(x_test, y_test)],verbose=100)
y_pred = xgb_model.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(r2)
print(rmse)

[0]	validation_0-rmse:35.75354
[100]	validation_0-rmse:14.98145
[200]	validation_0-rmse:14.45184
[300]	validation_0-rmse:14.17807
[400]	validation_0-rmse:14.01447
[499]	validation_0-rmse:13.87824
0.8607218265533447
13.878236685228096


In [81]:
rmse,r2
new_data_to_test = pd.DataFrame([{
  'PM2_5' : 15.8,
  'PM10' : 17.6,
  'CO' : 203.0,
  'NO2' : 11.9,
  'temperature' : 11.1,
  'precipitation' : 0.0,
  'relative_humidity' : 95,
  'wind_speed_num' : 3.2,
  'hours' : 0,
  'month' : 2,
  'day_of_week' : 6,
  
}])

predicted_value = xgb_model.predict(new_data_to_test).item()
predicted_value

59.66495895385742

In [67]:
import joblib
joblib.dump(xgb_model,'xgb_model_prediction.pkl')

['xgb_model_prediction.pkl']